# Lightweight causal DML pipeline

Uses your real `df`. Configure the workflow here; estimator mathematics are in [MATH.md](MATH.md). Nothing fabricates input data. Run from this repository directory.

In [ ]:
# Locate root when running inside notebooks/.
from pathlib import Path
import sys
REPO_ROOT = next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'causal_dml.py').exists())
sys.path.insert(0,str(REPO_ROOT))
import os
os.chdir(REPO_ROOT)  # historical example cells use repository-relative paths

In [ ]:
%pip install -r requirements-notebooks.txt

In [ ]:
from pathlib import Path
import pandas as pd
from dataclasses import replace
from causal_dml import CustomCausalDML, FinalStageConfig
from pipeline import PipelineConfig, run_pipeline

## Load real data

If `df` already exists, keep it. Otherwise replace the path below and uncomment `read_csv` or use your own BigQuery/parquet loading cell. Each row must have a valid attributed, mature outcome. This notebook cannot infer purchase timestamps or outcome-window maturity.

In [ ]:
# df = pd.read_csv('/path/to/real_pdp_views.csv')
# df = pd.read_parquet('/path/to/real_pdp_views.parquet')
if 'df' not in globals():
    raise RuntimeError('Load your actual dataframe as df before continuing.')
print(df.shape)

In [ ]:
cfg = PipelineConfig(
    outcome_col='revenue', group_col='user_id',
    x_cols=['industry', 'data_definition', 'region'],
    w_cols=['day', 'week', 'month', 'device', 'channel'],
    lead_col='lead_time', discount_col='discount_percentage',
    discount_input_is_percent=True,  # True: 7.7 is 7.7%; False: .077 is 7.7%
    max_discount=.45, max_lead=30, reference='0d|0%',
    min_action_rows=100, min_cohort_action_rows=100,
    sample_group_fraction=1.0,
    nuisance_y_params={'n_estimators': 100, 'learning_rate': .05, 'min_child_samples': 100},
    nuisance_t_params={'n_estimators': 100, 'learning_rate': .05, 'min_child_samples': 100},
    final_candidates=[
        FinalStageConfig(n_estimators=80, num_leaves=7, max_depth=3, min_child_samples=300),
        FinalStageConfig(n_estimators=160, learning_rate=.05, num_leaves=15, min_child_samples=200),
    ],
    scenarios=[
        {'name': 'all', 'filters': {}},
        {'name': 'lab_water', 'filters': {'data_definition': ['lab_water'], 'industry': [], 'region': []}},
    ],
)
# Alternative binary carousel treatment:
# cfg.treatment_col = 'carousel_seen'
# cfg.reference = 0
# cfg.categories = [0, 1]
# Review X/W for pretreatment status relative to this intervention.

In [ ]:
output_dir = Path('outputs')
fitted, report = run_pipeline(df, cfg, output_dir)
report

In [ ]:
if (output_dir/'preflight_actions.csv').exists():
    display(pd.read_csv(output_dir/'preflight_actions.csv'))
display(pd.read_json(output_dir/'validation_selection.json'))

In [ ]:
if (output_dir/'scenario_effects.csv').exists():
    display(pd.read_csv(output_dir/'scenario_effects.csv'))
if (output_dir/'cohort_heatmap.png').exists():
    from IPython.display import Image
    display(Image(filename=str(output_dir/'cohort_heatmap.png')))

## Reuse the estimator

`fitted.estimator` contains the independent model; `fitted.x_encoder` and `fitted.w_encoder` encode raw data. Keep the fitted encoders for future data. Only load trusted joblib files. `const_marginal_effect` always retains the treatment axis. Effects are T1 minus T0. The example below uses the first few rows for an API check, not an independent performance evaluation.

In [ ]:
rows = df.iloc[:20]
x, w = fitted.encode(rows)
print(fitted.estimator.const_marginal_effect(x).shape)
if cfg.discrete_treatment:
    labels = fitted.estimator.categories_
    print(fitted.estimator.effect(x, T0=labels[0], T1=labels[1]))
else:
    print(fitted.estimator.effect(x, T0=0., T1=1.))

## Interpretation

With log1p revenue, `shifted_geometric_pct` is `100*expm1(mean_effect)`: a change in geometric means of **revenue+1**, not automatically expected arithmetic revenue. The model has no forest confidence intervals. Opportunity and mitigation tables are exploratory and must be read with support and identification diagnostics. See [VALIDATION.md](VALIDATION.md).